# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuguda999/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Lane 2, confirmed** — Refresh / Content Opportunity Scoring, same lane since ML-02. Built on the same March 2026 slice and `is_declining` proxy verified in ML-04 (days 1–15 features → days 16–31 outcome, no future-window leakage).

**Two signal checks first, before trusting the rule:**

- **Signal 1 — CTR-vs-position** (behind FlyRank's `low_ctr_visible_page` / CTR-fix flag). Hypothesis: a page ranking well but earning less CTR than pages at that position typically earn is under-monetized and likelier to keep sliding. Bucket table + verdict below.
- **Signal 2 — Volume** (behind FlyRank's volume-gated `quick_win`-style logic). Hypothesis: low prior-volume pages are noisier — a *relative* decline threshold is easier to cross by chance on a small base — so volume is a legitimate confidence signal, not just a filter. Bucket table + verdict below.

**My rule, in plain words:** A page is worth reviewing first if it already earns real search volume and a workable position, but its click-through rate falls below what pages at that position typically earn. The size of that gap, scaled by volume, is the score.

**Score:** `score = imp_prev * max(0, expected_ctr_for_position_tier − ctr_prev)`, computed only for pages with `imp_prev >= 100` and `0 < avg_position_prev <= 20` (everyone else scores 0).

**Reason code (one):** `low_ctr_for_position` — fires when score > 0, otherwise `not_flagged`.

**Action label (one):** `review_ctr_fix` when it fires, `monitor` otherwise.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# find the repo root from wherever this kernel started (VS Code/Colab/CLI all differ)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

# Token order: env var -> Colab Secret -> prompt (last resort). Never hardcode — repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET http_timeout=300")  # seconds; default is 30s, too short for the heavier queries below
con.execute("SET http_retries=5")
con.execute("SET http_retry_wait_ms=1000")
con.execute("SET http_retry_backoff=2")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# same feature build + proxy label verified in ML-04 (w03_data_contract.ipynb)
feat = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_prev,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_prev,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_prev,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN report_date END) AS active_days_prev,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last
    FROM read_parquet('{MONTH}')
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) > 0
""").df()

feat["ctr_prev"] = feat["clk_prev"] / feat["imp_prev"]
feat["is_declining"] = (feat["imp_last"] < 0.8 * feat["imp_prev"]).astype(int)
feat = feat.dropna(subset=["avg_position_prev"]).reset_index(drop=True)

print(f"shape: {feat.shape[0]:,} rows  |  base decline rate: {feat['is_declining'].mean() * 100:.1f}%")

shape: 150,675 rows  |  base decline rate: 32.6%


### Signal check 1 — CTR-vs-position (behind the CTR-fix flag)

In [2]:
eligible = feat[(feat["imp_prev"] >= 100) & (feat["avg_position_prev"] > 0) & (feat["avg_position_prev"] <= 20)].copy()
eligible["position_tier"] = pd.cut(eligible["avg_position_prev"], bins=[0, 3, 10, 20], labels=["1-3", "4-10", "11-20"]).astype(str)

expected_ctr_by_tier = eligible.groupby("position_tier")["ctr_prev"].median()
print("median CTR per position tier (this becomes the rule's expected-CTR threshold):")
print(expected_ctr_by_tier)

eligible["expected_ctr"] = eligible["position_tier"].map(expected_ctr_by_tier.to_dict()).astype(float)
eligible["ctr_tier"] = np.where(eligible["ctr_prev"].values <= eligible["expected_ctr"].values, "low_ctr", "high_ctr")

signal1_bucket = eligible.groupby("ctr_tier").agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"))
print("\nSignal 1 bucket table (within position 1-20, imp_prev >= 100):")
print(signal1_bucket)
print("\nVerdict: CONFIRMED — low-CTR-for-position pages decline more often "
      f"({signal1_bucket.loc['low_ctr', 'decline_rate']*100:.1f}% vs {signal1_bucket.loc['high_ctr', 'decline_rate']*100:.1f}%, "
      f"n={signal1_bucket.loc['low_ctr', 'n']:,} vs n={signal1_bucket.loc['high_ctr', 'n']:,}).")

median CTR per position tier (this becomes the rule's expected-CTR threshold):
position_tier
1-3      0.002656
11-20    0.000980
4-10     0.001823
Name: ctr_prev, dtype: float64

Signal 1 bucket table (within position 1-20, imp_prev >= 100):
              n  decline_rate
ctr_tier                     
high_ctr  30974      0.204365
low_ctr   30976      0.336971

Verdict: CONFIRMED — low-CTR-for-position pages decline more often (33.7% vs 20.4%, n=30,976 vs n=30,974).


### Signal check 2 — volume (behind volume-gated quick-win logic)

In [3]:
feat["vol_tier"] = pd.qcut(feat["imp_prev"], q=3, labels=["low_vol", "mid_vol", "high_vol"])

signal2_bucket = feat.groupby("vol_tier", observed=True).agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"))
print("Signal 2 bucket table (volume tercile of imp_prev):")
print(signal2_bucket)
print("\nVerdict: CONFIRMED — decline rate drops monotonically as prior volume rises "
      f"({signal2_bucket.loc['low_vol', 'decline_rate']*100:.1f}% -> {signal2_bucket.loc['high_vol', 'decline_rate']*100:.1f}%, "
      f"n={signal2_bucket['n'].sum():,} total). Low-volume pages are noisier — a relative decline "
      "threshold is easier to cross by chance on a small base — so volume is a legitimate "
      "confidence signal, not just a filter.")

Signal 2 bucket table (volume tercile of imp_prev):
              n  decline_rate
vol_tier                     
low_vol   50507      0.401509
mid_vol   49977      0.293575
high_vol  50191      0.282959

Verdict: CONFIRMED — decline rate drops monotonically as prior volume rises (40.2% -> 28.3%, n=150,675 total). Low-volume pages are noisier — a relative decline threshold is easier to cross by chance on a small base — so volume is a legitimate confidence signal, not just a filter.


## 2. Build the ranked queue (writes the CSV)

Both signals CONFIRMED above, so the rule stands as written in section 1. Applying it to every row, ranking by score, and writing `work/outputs/baseline_action_score.csv`.

In [4]:
def expected_ctr_for_position(pos):
    if pos <= 3:
        return expected_ctr_by_tier["1-3"]
    if pos <= 10:
        return expected_ctr_by_tier["4-10"]
    return expected_ctr_by_tier["11-20"]

queue = feat.copy()
queue["expected_ctr"] = queue["avg_position_prev"].apply(expected_ctr_for_position)

rule_eligible = (queue["imp_prev"] >= 100) & (queue["avg_position_prev"] > 0) & (queue["avg_position_prev"] <= 20)
gap = np.where(rule_eligible, queue["expected_ctr"] - queue["ctr_prev"], 0.0)
queue["score"] = np.where(rule_eligible, queue["imp_prev"] * np.clip(gap, 0, None), 0.0)
queue["reason_code"] = np.where(queue["score"] > 0, "low_ctr_for_position", "not_flagged")
queue["action"] = np.where(queue["score"] > 0, "review_ctr_fix", "monitor")

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

base_rate = queue["is_declining"].mean()
print(f"flagged rows (score > 0): {(queue['score'] > 0).sum():,} / {len(queue):,}")
print(f"base rate (overall decline rate): {base_rate:.3f}")
for k in [10, 20, 50, 100]:
    p_at_k = queue.head(k)["is_declining"].mean()
    print(f"precision@{k}: {p_at_k:.3f}")

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["client_hash_id", "content_hash_id", "imp_prev", "avg_position_prev", "ctr_prev",
            "expected_ctr", "active_days_prev", "score", "reason_code", "action"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"\nwrote work/outputs/baseline_action_score.csv  ({len(queue):,} rows)")

flagged rows (score > 0): 30,974 / 150,675
base rate (overall decline rate): 0.326
precision@10: 0.600
precision@20: 0.600
precision@50: 0.460
precision@100: 0.460

wrote work/outputs/baseline_action_score.csv  (150,675 rows)


## 3. Top-10 review

*(This card asks for ten, not the skeleton title's twenty — a top-20 pass is the optional stretch in the sibling `w04_signal_audit.ipynb`.)*

For each of the top 10: the action, why it's there, and what would make it wrong.

In [5]:
top10 = queue.head(10)

for rank, row in enumerate(top10.itertuples(), start=1):
    print(f"#{rank}  {row.client_hash_id} / {row.content_hash_id}")
    print(f"    action={row.action}  reason={row.reason_code}  score={row.score:.1f}")
    print(f"    position={row.avg_position_prev:.1f}  ctr={row.ctr_prev:.4f}  expected_ctr={row.expected_ctr:.4f}  "
          f"imp_prev={row.imp_prev:,.0f}  active_days_prev={row.active_days_prev}/15")
    print(f"    why it's here: real volume + workable position, but CTR sits well below what "
          f"position {row.avg_position_prev:.0f} typically earns.")
    if row.active_days_prev < 15:
        wrong_note = f"only {row.active_days_prev}/15 active days behind this read — thin evidence, could be a mid-window tracking gap, not a real CTR problem."
    else:
        wrong_note = "wrong if this query's intent is branded/navigational (structurally low CTR despite good position) or the SERP snippet already changed and hasn't rolled through fully."
    print(f"    what would make it wrong: {wrong_note}")
    print(f"    outcome (for my own read only, not used in the rule): is_declining={row.is_declining}")
    print()

#1  client_62f4a7e64f5e0096 / content_34a70fea29d15f24
    action=review_ctr_fix  reason=low_ctr_for_position  score=177.6
    position=2.8  ctr=0.0002  expected_ctr=0.0027  imp_prev=73,639  active_days_prev=15/15
    why it's here: real volume + workable position, but CTR sits well below what position 3 typically earns.
    what would make it wrong: wrong if this query's intent is branded/navigational (structurally low CTR despite good position) or the SERP snippet already changed and hasn't rolled through fully.
    outcome (for my own read only, not used in the rule): is_declining=0

#2  client_73cda7b4e4f265ea / content_9c057b66c30a3abb
    action=review_ctr_fix  reason=low_ctr_for_position  score=152.7
    position=8.6  ctr=0.0000  expected_ctr=0.0018  imp_prev=83,772  active_days_prev=15/15
    why it's here: real volume + workable position, but CTR sits well below what position 9 typically earns.
    what would make it wrong: wrong if this query's intent is branded/navigational 

## 4. Weak picks + leakage check

**Weak pick:** the #1-ranked row scored highest precisely because it has the largest volume × CTR-gap — but its actual outcome was `is_declining = 0`. Highest score does not guarantee the outcome; a large, stable page can sit well below its "expected" CTR indefinitely (branded query, feature snippet stealing clicks, intent mismatch) without ever declining. Score ranks *risk of an under-monetized gap*, not *certainty of decline* — that's exactly what precision@10 = 0.60 (vs base rate 0.326) already says: better than chance, not proof.

**Leakage check:** the score formula only reads `imp_prev`, `avg_position_prev`, and `ctr_prev` (= `clk_prev / imp_prev`) — all closed over days 1–15. `imp_last` and `is_declining` never enter the score; they're read afterward, only to evaluate the rule (precision@K), never to build it. Confirmed below.

In [6]:
top1 = queue.iloc[0]
print(f"#1 ranked row: score={top1['score']:.1f}, is_declining={top1['is_declining']}  <- confirms the weak-pick claim above")

score_inputs = {"imp_prev", "avg_position_prev", "ctr_prev"}
label_derived = {"imp_last", "is_declining"}
print(f"\nscore formula inputs: {sorted(score_inputs)}")
print(f"label-derived columns (never in the formula): {sorted(label_derived)}")
assert score_inputs.isdisjoint(label_derived), "LEAKAGE: a label-derived column reached the score formula"
print("assert passed: no overlap between score inputs and label-derived columns.")

#1 ranked row: score=177.6, is_declining=0  <- confirms the weak-pick claim above

score formula inputs: ['avg_position_prev', 'ctr_prev', 'imp_prev']
label-derived columns (never in the formula): ['imp_last', 'is_declining']
assert passed: no overlap between score inputs and label-derived columns.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.